# Churn Prediction

The executable training and evaluation workflow is in `04_churn_prediction.py`. Run the cell below to load the processed datasets, build historical features, compare the four classifiers, and report held-out results.

## Method and interpretation

Features are frozen at 2025-10-02, before the earliest churn event; customers registered after that date are excluded. All event-based features are aggregated only from records on or before the cutoff. Churn label dates/reasons, subscription cancellation/status fields, and post-cutoff activity are not used as predictors.

### Code flow

1. Read the cleaned customer and churn-label tables; set the feature cutoff to 2025-10-02, before the first observed churn.
2. Build one row per eligible customer, aggregating pre-cutoff viewing, subscription, payment, support-ticket, and feedback history.
3. Split the rows into stratified training (80%) and test (20%) sets. Impute missing values, scale numeric fields, and one-hot encode categories inside each model pipeline.
4. Compare Logistic Regression, Random Forest, Gradient Boosting, and XGBoost when its dependencies are available. Balanced sample weights address the lower churn-class count.
5. Run stratified 5-fold cross-validation on training data; select the candidate by average precision (PR-AUC), with ROC-AUC as a tie-breaker.
6. Choose an F1-maximizing threshold from out-of-fold training predictions, fit the selected model on all training rows, and evaluate it on the held-out test set.
7. Print accuracy, precision, recall, F1, ROC-AUC, average precision, the confusion matrix, and a written model-selection/retention-offer explanation.


For retention offers, increasing recall catches more customers likely to churn but also spends offers on customers who would have stayed. Increasing precision reduces those wasted offers but misses more churners. The F1 threshold is a transparent baseline, not an economic optimum; tune it to offer cost, campaign capacity, expected customer value, and measured incremental retention.

On macOS, XGBoost also needs the OpenMP runtime (`brew install libomp`). If that runtime is unavailable, the script reports the reason and compares the other candidates.

In [1]:
# Locate the script whether this notebook is opened from the repository root
# or launched with the notebook directory as the current working directory.
from pathlib import Path
import runpy

# Run the same complete, documented workflow used from the command line.
script_path = Path("notebooks/04_churn_prediction.py")
if not script_path.is_file():
    script_path = Path("04_churn_prediction.py")
if not script_path.is_file():
    raise FileNotFoundError("Could not locate notebooks/04_churn_prediction.py")

runpy.run_path(str(script_path), run_name="__main__")

XGBoost candidate skipped: install the xgboost package to enable it.
Feature cutoff: 2025-10-02
Eligible customers: 7,277
Churn label counts (0 = retained, 1 = churned):
churned
0    5500
1    1777
Churn rate: 24.4%

Cross-validating Logistic Regression...
Cross-validating Random Forest...
Cross-validating Gradient Boosting...
5-fold cross-validation on training data:
              model  average_precision  roc_auc  precision  recall    f1  accuracy
  Gradient Boosting              0.317    0.582      0.297   0.455 0.358     0.604
Logistic Regression              0.310    0.579      0.289   0.507 0.368     0.576
      Random Forest              0.308    0.572      0.385   0.110 0.172     0.740

Selected model by CV average precision: Gradient Boosting

Held-out test metrics at the training-only F1 threshold:
         threshold: 0.460
          accuracy: 0.455
         precision: 0.271
            recall: 0.730
                f1: 0.396
           roc_auc: 0.592
 average_precision: 0.31

{'__name__': '__main__',
 '__doc__': 'Train and evaluate a leakage-aware customer churn classifier.',
 '__package__': '',
 '__loader__': None,
 '__spec__': None,
 '__file__': '04_churn_prediction.py',
 '__cached__': None,
 '__builtins__': {'__name__': 'builtins',
  '__doc__': "Built-in functions, exceptions, and other objects.\n\nNoteworthy: None is the `nil' object; Ellipsis represents `...' in slices.",
  '__package__': '',
  '__loader__': _frozen_importlib.BuiltinImporter,
  '__spec__': ModuleSpec(name='builtins', loader=<class '_frozen_importlib.BuiltinImporter'>, origin='built-in'),
  '__build_class__': <function __build_class__>,
  '__import__': <function __import__>,
  'abs': <function abs(x, /)>,
  'all': <function all(iterable, /)>,
  'any': <function any(iterable, /)>,
  'ascii': <function ascii(obj, /)>,
  'bin': <function bin(number, /)>,
  'breakpoint': <function breakpoint>,
  'callable': <function callable(obj, /)>,
  'chr': <function chr(i, /)>,
  'compile': <function c